## Preprocessing

In [51]:
# imports and setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from pathlib import Path

from utils import (
    load_resume_data, 
    load_job_skills, 
    load_job_summary, 
    load_linkedin_jobs,
    ensure_processed_dir,
    sample_resumes_stratified, 
    sample_jobs_stratified
)

from preprocessing import PreprocessText
import json

In [21]:
# Path to preprocessed data directory
DATA_DIR = '../data'
processed_dir = ensure_processed_dir(DATA_DIR)

print(f"Processed data will be saved to: {processed_dir}")

Processed data will be saved to: ../data\processed


In [22]:
# Load data
print("\nLoading raw datasets...")

resumes_df = load_resume_data(DATA_DIR)
job_skills_df = load_job_skills(DATA_DIR)
job_summary_df = load_job_summary(DATA_DIR)
linkedin_jobs_df = load_linkedin_jobs(DATA_DIR)

print(f"✓ Resumes: {len(resumes_df)} records")
print(f"✓ Job skills: {len(job_skills_df)} records")
print(f"✓ Job summaries: {len(job_summary_df)} records")
print(f"✓ LinkedIn jobs: {len(linkedin_jobs_df)} records")


Loading raw datasets...
✓ Resumes: 2484 records
✓ Job skills: 1296381 records
✓ Job summaries: 1297332 records
✓ LinkedIn jobs: 1348454 records


In [23]:
print("Selecting useful columns and merging job datasets...")

# Keep only the most useful resume columns for now
full_resumes = resumes_df[['ID', 'Resume_str', 'Category']].copy()

# Merge the three job tables on job_link
jobs_merged = linkedin_jobs_df.merge(job_summary_df, on='job_link', how='left')
jobs_merged = jobs_merged.merge(job_skills_df, on='job_link', how='left')

# Keep useful job columns
full_jobs = jobs_merged[
    [
        'job_link',
        'job_title',
        'company',
        'job_location',
        'search_city',
        'search_country',
        'search_position',
        'job_level',
        'job_type',
        'job_summary',
        'job_skills'
    ]
].copy()

print(f"✓ Full resumes selected: {len(full_resumes):,} records")
print(f"✓ Full jobs merged: {len(full_jobs):,} records")

print("\nMerged job columns:")
print(full_jobs.columns.tolist())

Selecting useful columns and merging job datasets...
✓ Full resumes selected: 2,484 records
✓ Full jobs merged: 1,348,454 records

Merged job columns:
['job_link', 'job_title', 'company', 'job_location', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_summary', 'job_skills']


In [24]:
# Clean resume data
print("Cleaning resume text...")

preprocessor = PreprocessText(remove_stopwords=False)

full_resumes['Resume_cleaned'] = full_resumes['Resume_str'].apply(
    lambda x: preprocessor.clean_text(x) if isinstance(x, str) else ""
)

print(f"✓ Cleaned {len(full_resumes):,} resume texts")

print("\nSample original resume (first 200 chars):")
print(full_resumes['Resume_str'].iloc[0][:200])

print("\nSample cleaned resume (first 200 chars):")
print(full_resumes['Resume_cleaned'].iloc[0][:200])

Cleaning resume text...
✓ Cleaned 2,484 resume texts

Sample original resume (first 200 chars):
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Resp

Sample cleaned resume (first 200 chars):
hr administrator marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management. respected builder and lead


In [25]:
# Clean job data
print("Cleaning job text fields...")

full_jobs['job_title_cleaned'] = full_jobs['job_title'].apply(
    lambda x: preprocessor.clean_text(x) if isinstance(x, str) else ""
)

full_jobs['job_summary_cleaned'] = full_jobs['job_summary'].apply(
    lambda x: preprocessor.clean_text(x) if isinstance(x, str) else ""
)

full_jobs['job_skills_cleaned'] = full_jobs['job_skills'].apply(
    lambda x: preprocessor.clean_text(x) if isinstance(x, str) else ""
)

full_jobs['job_combined_text'] = (
    full_jobs['job_title'].fillna('').astype(str) + ' ' +
    full_jobs['job_summary'].fillna('').astype(str) + ' ' +
    full_jobs['job_skills'].fillna('').astype(str)
).apply(preprocessor.clean_text)

print(f"✓ Cleaned {len(full_jobs):,} job records")

print("\nSample cleaned job title:")
print(full_jobs['job_title_cleaned'].iloc[0])

print("\nSample cleaned job summary (first 200 chars):")
print(full_jobs['job_summary_cleaned'].iloc[0][:200])

print("\nSample cleaned job skills:")
print(full_jobs['job_skills_cleaned'].iloc[0])

Cleaning job text fields...
✓ Cleaned 1,348,454 job records

Sample cleaned job title:
account executive - dispensing norcal northern nevada - becton dickinson

Sample cleaned job summary (first 200 chars):
responsibilities job description summary job description we are the makers of possible bd is one of the largest global medical technology companies in the world. advancing the world of health is our p

Sample cleaned job skills:
medical equipment sales, key competitors, terminology, technology, trends, challenges, reimbursement, government regulation, bd offerings, pipeline management, opportunity planning, solution sales process, customer relationships, collaboration, negotiation, internal and external customer satisfaction, trust building, sales opportunities, new solution introduction, pharmacy, it, nursing, csuite, bachelor s degree, travel, driver s license


In [26]:
# Check for missing values
print("Checking missing values...")

print("\nResume null values:")
print(full_resumes[['ID', 'Resume_str', 'Category', 'Resume_cleaned']].isnull().sum())

print("\nJob null values:")
print(full_jobs[
    [
        'job_title',
        'job_summary',
        'job_skills',
        'job_title_cleaned',
        'job_summary_cleaned',
        'job_skills_cleaned',
        'job_combined_text'
    ]
].isnull().sum())

# Fill missing values
full_resumes['Resume_str'] = full_resumes['Resume_str'].fillna("")
full_resumes['Resume_cleaned'] = full_resumes['Resume_cleaned'].fillna("")
full_resumes['Category'] = full_resumes['Category'].fillna("Unknown")

full_jobs['job_title'] = full_jobs['job_title'].fillna("")
full_jobs['job_summary'] = full_jobs['job_summary'].fillna("")
full_jobs['job_skills'] = full_jobs['job_skills'].fillna("")
full_jobs['job_title_cleaned'] = full_jobs['job_title_cleaned'].fillna("")
full_jobs['job_summary_cleaned'] = full_jobs['job_summary_cleaned'].fillna("")
full_jobs['job_skills_cleaned'] = full_jobs['job_skills_cleaned'].fillna("")
full_jobs['job_combined_text'] = full_jobs['job_combined_text'].fillna("")

print("\n✓ Missing values handled")

Checking missing values...

Resume null values:
ID                0
Resume_str        0
Category          0
Resume_cleaned    0
dtype: int64

Job null values:
job_title                  0
job_summary            51122
job_skills             54158
job_title_cleaned          0
job_summary_cleaned        0
job_skills_cleaned         0
job_combined_text          0
dtype: int64

✓ Missing values handled


In [27]:
# Check for duplicates
print("Checking duplicates...")

print(f"Resume row duplicates: {full_resumes.duplicated().sum():,}")
print(f"Resume text duplicates: {full_resumes['Resume_str'].duplicated().sum():,}")

print(f"Job row duplicates: {full_jobs.duplicated().sum():,}")
print(f"Duplicate job links: {full_jobs['job_link'].duplicated().sum():,}")

print("\nSample cleaned resumes:")
display(full_resumes[['ID', 'Category', 'Resume_cleaned']].head(3))

print("\nSample cleaned jobs:")
display(full_jobs[['job_title', 'job_summary_cleaned', 'job_skills_cleaned', 'job_combined_text']].head(3))

Checking duplicates...
Resume row duplicates: 0
Resume text duplicates: 2
Job row duplicates: 0
Duplicate job links: 0

Sample cleaned resumes:


,ID,Category,Resume_cleaned
0,16852973,HR,hr administrator marketing associate hr admini...
1,22323967,HR,"hr specialist, us hr operations summary versat..."
2,33176873,HR,hr director summary over 20 years experience i...



Sample cleaned jobs:


,job_title,job_summary_cleaned,job_skills_cleaned,job_combined_text
0,Account Executive - Dispensing (NorCal/Norther...,responsibilities job description summary job d...,"medical equipment sales, key competitors, term...",account executive - dispensing norcal northern...
1,Registered Nurse - RN Care Manager,employment type full time shift description po...,"nursing, bachelor of science in nursing, maste...",registered nurse - rn care manager employment ...
2,RESTAURANT SUPERVISOR - THE FORKLIFT,job details description what you ll do as a fo...,"restaurant operations management, inventory ma...",restaurant supervisor - the forklift job detai...


In [28]:
# Save processed datasets
print("Saving processed datasets...")

resume_output_path = f"{processed_dir}/resumes_cleaned.csv"
jobs_output_path = f"{processed_dir}/jobs_merged_cleaned.csv"

full_resumes.to_csv(resume_output_path, index=False)
full_jobs.to_csv(jobs_output_path, index=False)

print(f"✓ Saved resumes to: {resume_output_path}")
print(f"✓ Saved jobs to: {jobs_output_path}")

Saving processed datasets...
✓ Saved resumes to: ../data\processed/resumes_cleaned.csv
✓ Saved jobs to: ../data\processed/jobs_merged_cleaned.csv


In [29]:
# Print summary 
print("Preprocessing summary")
print("-" * 40)
print(f"Total resumes: {len(full_resumes):,}")
print(f"Total jobs: {len(full_jobs):,}")
print(f"Resume categories: {full_resumes['Category'].nunique():,}")
print(f"Missing resume text after cleaning: {(full_resumes['Resume_cleaned'] == '').sum():,}")
print(f"Missing job combined text after cleaning: {(full_jobs['job_combined_text'] == '').sum():,}")

Preprocessing summary
----------------------------------------
Total resumes: 2,484
Total jobs: 1,348,454
Resume categories: 24
Missing resume text after cleaning: 1
Missing job combined text after cleaning: 3


In [56]:
# Build smaller sample datasets for testing
print("Creating stratified subsets...")

# Stratified resume sampling (balanced by Category)
resumes_subset = sample_resumes_stratified(full_resumes, n_per_category=30)

# Random job sampling (large dataset → just sample)
jobs_subset = sample_jobs_stratified(full_jobs, n_samples=3000)

print(f"✓ Resume subset: {len(resumes_subset):,} records")
print(f"✓ Job subset: {len(jobs_subset):,} records")

# save subsets
resumes_subset.to_csv(f"{processed_dir}/resumes_subset.csv", index=False)
jobs_subset.to_csv(f"{processed_dir}/jobs_subset.csv", index=False)

print("✓ Saved subset datasets")

Creating stratified subsets...
✓ Resume subset: 712 records
✓ Job subset: 3,000 records
✓ Saved subset datasets


In [57]:
print("\nResume category distribution (subset):")
print(resumes_subset['Category'].value_counts().head(10))


Resume category distribution (subset):
Category
ACCOUNTANT              30
ADVOCATE                30
AGRICULTURE             30
APPAREL                 30
ARTS                    30
AUTOMOBILE              30
AVIATION                30
BANKING                 30
BUSINESS-DEVELOPMENT    30
CHEF                    30
Name: count, dtype: int64
